In [1]:
# ── CELL 1: Imports ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
# ── CELL 2: Load Data (Kaggle paths) ─────────────────────────────────────────
df_train = pd.read_csv('titanic_data/train.csv')
df_test  = pd.read_csv('titanic_data/test.csv')

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ── CELL 3: YOUR Feature Engineering — Train ─────────────────────────────────
def engineer_features(df):
    df = df.copy()
    
    # Drop string/identifier columns that will break matrix multiplication
    df.drop(['Cabin', 'Name', 'Ticket'], axis=1, inplace=True, errors='ignore')

    # Embarked
    df['Embarked'].fillna('S', inplace=True)
    df.replace({"Embarked": {"S": 0, "C": 1, "Q": 2}}, inplace=True)

    # Sex
    df.replace({"Sex": {"male": 0, "female": 1}}, inplace=True)

    # Age — fill by sex mean
    male_mean   = 30.7266445916114
    female_mean = 27.9157081226057
    df.loc[(df['Sex'] == 0) & (df['Age'].isnull()), 'Age'] = male_mean
    df.loc[(df['Sex'] == 1) & (df['Age'].isnull()), 'Age'] = female_mean
    df['Age'].fillna(df['Age'].median(), inplace=True)

    # Age group
    age        = [0, 5, 15, 25, 30, 35, 45, 50, 200]
    age_label  = ['0-5','5-15','15-25','25-30','30-35','35-40','45-50','>50']
    df['age_group']      = pd.cut(df['Age'], age, labels=age_label)
    df['age_group_code'] = df['age_group'].cat.codes

    # Fare group
    df['Fare'].fillna(df['Fare'].median(), inplace=True)
    price       = [0, 10, 30, 35, 80, 1000]
    price_label = ['0-10','10-30','30-35','35-80','>80']
    df['price_group']      = pd.cut(df['Fare'], price, labels=price_label)
    df['price_group_code'] = df['price_group'].cat.codes

    # Family size
    df['SumPeople'] = df['SibSp'].astype(int) + df['Parch'].astype(int) + 1
    
    # Drop the categorical columns we replaced with codes
    df.drop(['age_group', 'price_group'], axis=1, inplace=True, errors='ignore')

    # SCALE THE FEATURES so the tanh gradients don't vanish
    scaler = StandardScaler()
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    # We shouldn't scale the target variable 'Survived' if it's in the training set
    cols_to_scale = [col for col in numeric_cols if col not in ['Survived', 'PassengerId']]
    
    df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

    return df

df_train = engineer_features(df_train)
df_test  = engineer_features(df_test)

In [4]:
# ── CELL 4: Prepare arrays ───────────────────────────────────────────────────
FEATURES = ['Pclass', 'Sex', 'age_group_code', 'price_group_code',
            'SumPeople', 'SibSp', 'Parch', 'Embarked']

X_train = df_train[FEATURES].values.astype(float)
y_train = df_train['Survived'].values.astype(float)
X_test  = df_test[FEATURES].values.astype(float)

# Normalise to [0,1] using train statistics
X_min = X_train.min(axis=0)
X_max = X_train.max(axis=0)
X_train = (X_train - X_min) / (X_max - X_min + 1e-8)
X_test  = (X_test  - X_min) / (X_max - X_min + 1e-8)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

X_train: (891, 8)  |  X_test: (418, 8)


In [5]:
import numpy as np
from tqdm import tqdm

# ── CELL 5: Predictive Coding Network ────────────────────────────────────────
class PredictiveCodingNetwork:
    """
    Hierarchical Predictive Coding Network (Rao & Ballard, 1999).
    - Top-down connections generate predictions.
    - Prediction errors propagate bottom-up.
    - Activities minimised via iterative inference.
    - Weight updates are LOCAL — no global backpropagation.
    """
    def __init__(self, layer_sizes, lr_weights=0.001, lr_activities=0.05,
                 n_inference_steps=30):
        self.sizes   = layer_sizes
        self.lr_w    = lr_weights
        self.lr_a    = lr_activities
        self.n_inf   = n_inference_steps
        self.n_layers = len(layer_sizes)

        # Top-down weight matrices: W[l] predicts layer l from layer l+1
        self.W = [np.random.randn(layer_sizes[l], layer_sizes[l+1]) * 0.1
                  for l in range(self.n_layers - 1)]
        self.b = [np.zeros(layer_sizes[l])
                  for l in range(self.n_layers - 1)]

    def _f(self, x):   return np.tanh(x)
    def _df(self, x):  return 1.0 - np.tanh(x) ** 2
    def _sig(self, x): return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

    def _predict_down(self, r_above, l):
        return self._f(self.W[l] @ r_above + self.b[l])

    def _infer(self, x, y=None):
        r = [np.zeros(s) for s in self.sizes]
        r[0] = x.copy()

        # Warm-start: bottom-up sweep
        for l in range(self.n_layers - 1):
            r[l+1] = self._f(self.W[l].T @ r[l])

        # Pin label if training
        if y is not None:
            r[-1] = np.array([y])

        # Inference loop — minimise prediction error in hidden layers
        for _ in range(self.n_inf):
            errors = [r[l] - self._predict_down(r[l+1], l)
                      for l in range(self.n_layers - 1)]
            
            for l in range(1, self.n_layers - 1):
                # Corrected index mapping to match the local gradient of energy
                grad = (errors[l] 
                        - self.W[l-1].T @ (errors[l-1] * self._df(self.W[l-1] @ r[l] + self.b[l-1])))
                r[l] -= self.lr_a * grad

        errors = [r[l] - self._predict_down(r[l+1], l)
                  for l in range(self.n_layers - 1)]
        return r, errors

    def _update(self, r, errors):
        for l in range(self.n_layers - 1):
            self.W[l] -= self.lr_w * np.outer(errors[l], r[l+1])
            self.b[l] -= self.lr_w * errors[l]

    def fit(self, X, y, epochs=100, verbose=True):
        for epoch in range(epochs):
            idx = np.random.permutation(len(X))
            correct = 0
            
            # Use tqdm to track estimated time per epoch
            loop = tqdm(idx, disable=not verbose, desc=f"Epoch {epoch+1:3d}/{epochs}")
            
            for i in loop:
                r, errors = self._infer(X[i], y[i])
                self._update(r, errors)
                prob = self._sig(self.W[-1].T @ r[-2])
                correct += int((prob[0] > 0.5) == bool(y[i]))
            
            if verbose:
                # Append accuracy to the progress bar string
                loop.set_postfix(Acc=f"{correct/len(X):.4f}")
                
    def predict(self, X):
        return np.array([
            int(self._sig(self.W[-1].T @ self._infer(x)[0][-2])[0] > 0.5)
            for x in X
        ])

In [6]:
# ── CELL 6: Train ────────────────────────────────────────────────────────────
np.random.seed(42)
n_feat = X_train.shape[1]

model = PredictiveCodingNetwork(
    layer_sizes=[n_feat, 32, 16, 1],
    lr_weights=0.005,
    lr_activities=0.05,
    n_inference_steps=30
)

print(f"Architecture: {n_feat} → 32 → 16 → 1  |  NO backpropagation\n")
model.fit(X_train, y_train, epochs=1, verbose=True)



Architecture: 8 → 32 → 16 → 1  |  NO backpropagation



Epoch   1/1:   0%|          | 0/891 [00:00<?, ?it/s]

Epoch   1/1:   9%|▊         | 77/891 [00:00<00:01, 764.68it/s]

Epoch   1/1:  17%|█▋        | 154/891 [00:00<00:00, 758.32it/s]

Epoch   1/1:  26%|██▌       | 230/891 [00:00<00:00, 754.53it/s]

...

Epoch   1/1:  68%|██████▊   | 608/891 [00:00<00:00, 746.16it/s]

Epoch   1/1:  77%|███████▋  | 684/891 [00:00<00:00, 748.79it/s]

Epoch   1/1:  85%|████████▌ | 759/891 [00:01<00:00, 749.08it/s]

Epoch   1/1:  94%|█████████▎| 835/891 [00:01<00:00, 749.46it/s]

Epoch   1/1: 100%|██████████| 891/891 [00:01<00:00, 745.30it/s]

In [7]:
# ── CELL 7: Evaluate ─────────────────────────────────────────────────────────
train_acc = np.mean(model.predict(X_train) == y_train)
print(f"\nFinal Train Accuracy: {train_acc:.4f}")


Final Train Accuracy: 0.6162


In [8]:
# ── CELL 8: Submission ───────────────────────────────────────────────────────
test_preds = model.predict(X_test)
submission = pd.DataFrame({
    "PassengerId": df_test["PassengerId"],
    "Survived":    test_preds
})
submission.to_csv("submission_predictive_coding.csv", index=False)
print(f"\nSaved: submission_predictive_coding.csv")
print(f"Predicted survivors: {test_preds.sum()} / {len(test_preds)}")
print(submission.head(10))


Saved: submission_predictive_coding.csv
Predicted survivors: 0 / 418
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         0
5          897         0
6          898         0
7          899         0
8          900         0
9          901         0
